In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Mobile Reviews Sentiment null .csv')

In [3]:
df.head()

,review_id,customer_name,age,brand,model,price_usd,price_local,currency,exchange_rate_to_usd,rating,...,language,review_date,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,helpful_votes,source
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,₹27996.73,INR,83.00,2.0,...,Hindi,11/6/2023,True,1,1,3,2,1,1,Amazon
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,R$1754.35,BRL,5.70,4.0,...,Portuguese,3/30/2023,True,3,2,4,3,2,5,Flipkart
2,3,Pahal Balay,27,Google,Pixel 6,864.53,₹71755.99,INR,83.00,4.0,...,Hindi,12/7/2022,True,3,5,3,2,4,8,AliExpress
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,د.إ2425.65,AED,3.67,3.0,...,English,3/11/2025,False,1,3,2,1,2,3,Amazon
4,5,Yago Leão,38,Motorola,Edge 50,792.13,R$4515.14,BRL,5.70,3.0,...,Portuguese,9/29/2023,True,3,3,2,2,1,0,BestBuy


In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.isnull().sum()

review_id                  0
customer_name              0
age                        0
brand                      0
model                      0
price_usd               2450
price_local             2431
currency                   0
exchange_rate_to_usd       0
rating                  2453
sentiment               2445
country                    0
language                   0
review_date                0
verified_purchase          0
battery_life_rating        0
camera_rating              0
performance_rating         0
design_rating              0
display_rating             0
helpful_votes              0
source                  2448
dtype: int64

In [6]:
df['rating'] = df['rating'].fillna(df['rating'].median())
df['source'] = df['source'].fillna(df['source'].mode()[0])

In [7]:
df['price_usd'] = df.groupby('brand')['price_usd'].transform(lambda x: x.fillna(x.median()))

In [8]:
def fill_sentiment(row):
    if pd.isnull(row['sentiment']):
        if row['rating'] >= 4:
            return 'Positive'
        elif row['rating'] == 3:
            return 'Neutral'
        else:
            return 'Negative'
    return row['sentiment']

df['sentiment'] = df.apply(fill_sentiment, axis=1)

In [9]:
df.drop(columns=['price_local'], inplace=True)

In [10]:
df.to_csv('Mobile Reviews Sentiment Cleaned.csv', index=False)

In [11]:
df_clean = pd.read_csv('Mobile Reviews Sentiment Cleaned.csv')
df_clean.head()
df_clean.columns.tolist()

['review_id',
 'customer_name',
 'age',
 'brand',
 'model',
 'price_usd',
 'currency',
 'exchange_rate_to_usd',
 'rating',
 'sentiment',
 'country',
 'language',
 'review_date',
 'verified_purchase',
 'battery_life_rating',
 'camera_rating',
 'performance_rating',
 'design_rating',
 'display_rating',
 'helpful_votes',
 'source']

In [12]:
df_clean.isnull().sum()

review_id               0
customer_name           0
age                     0
brand                   0
model                   0
price_usd               0
currency                0
exchange_rate_to_usd    0
rating                  0
sentiment               0
country                 0
language                0
review_date             0
verified_purchase       0
battery_life_rating     0
camera_rating           0
performance_rating      0
design_rating           0
display_rating          0
helpful_votes           0
source                  0
dtype: int64

**Encoding**

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MultiLabelBinarizer

dfe = pd.read_csv('Mobile Reviews Sentiment Cleaned.csv')

In [14]:
le = LabelEncoder()

for col in ['brand', 'model', 'source', 'currency', 'language', 'sentiment']:
    dfe[col + '_encoded'] = le.fit_transform(dfe[col].astype(str))

dfe[['brand', 'brand_encoded', 'model', 'model_encoded', 'sentiment', 'sentiment_encoded']].head()

,brand,brand_encoded,model,model_encoded,sentiment,sentiment_encoded
0,Realme,4,Realme 12 Pro,15,Negative,0
1,Realme,4,Realme 12 Pro,15,Positive,2
2,Google,1,Pixel 6,10,Positive,2
3,Xiaomi,6,Redmi Note 13,17,Positive,2
4,Motorola,2,Edge 50,0,Neutral,1


In [15]:
print(dfe['source'].unique()[:10])
print(dfe['language'].unique()[:10])

<ArrowStringArray>
['Amazon', 'Flipkart', 'AliExpress', 'BestBuy', 'eBay']
Length: 5, dtype: str
<ArrowStringArray>
['Hindi', 'Portuguese', 'English', 'German']
Length: 4, dtype: str


In [16]:
mlb = MultiLabelBinarizer()

dfe['language_split'] = dfe['language'].apply(lambda x: str(x).split(','))
lang_encoded = mlb.fit_transform(dfe['language_split'])
lang_df = pd.DataFrame(lang_encoded, columns=['lang_' + c.strip() for c in mlb.classes_])

dfe = pd.concat([dfe, lang_df], axis=1)
dfe.drop(columns=['language_split'], inplace=True)
print(lang_df.head())

   lang_English  lang_German  lang_Hindi  lang_Portuguese
0             0            0           1                0
1             0            0           0                1
2             0            0           1                0
3             1            0           0                0
4             0            0           0                1


In [ ]:
dfe.drop(columns=['currency', 'language', 'sentiment', 'price_local'], inplace=True)

In [ ]:
for col in ['verified_purchase', 'country']:
    dfe[col + '_encoded'] = le.fit_transform(dfe[col].astype(str))

dfe['review_date'] = pd.to_datetime(dfe['review_date'], errors='coerce')
dfe['review_year'] = dfe['review_date'].dt.year
dfe['review_month'] = dfe['review_date'].dt.month

dfe.drop(columns=['verified_purchase','country','customer_name','review_date'], inplace=True)

print(dfe.shape)
print(dfe.columns.tolist())

(50000, 25)
['review_id', 'age', 'price_usd', 'exchange_rate_to_usd', 'rating', 'battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating', 'helpful_votes', 'brand_encoded', 'model_encoded', 'source_encoded', 'currency_encoded', 'language_encoded', 'sentiment_encoded', 'lang_English', 'lang_German', 'lang_Hindi', 'lang_Portuguese', 'verified_purchase_encoded', 'country_encoded', 'review_year', 'review_month']


In [35]:
print(dfe.shape)
print(dfe.columns.tolist())


(50000, 25)
['review_id', 'age', 'price_usd', 'exchange_rate_to_usd', 'rating', 'battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating', 'helpful_votes', 'brand_encoded', 'model_encoded', 'source_encoded', 'currency_encoded', 'language_encoded', 'sentiment_encoded', 'lang_English', 'lang_German', 'lang_Hindi', 'lang_Portuguese', 'verified_purchase_encoded', 'country_encoded', 'review_year', 'review_month']


In [36]:
dfe.to_csv('Mobile Reviews Sentiment Encoded.csv', index=False)

In [20]:
dfe.head()

,review_id,customer_name,age,brand,model,price_usd,currency,exchange_rate_to_usd,rating,sentiment,...,brand_encoded,model_encoded,source_encoded,currency_encoded,language_encoded,sentiment_encoded,lang_English,lang_German,lang_Hindi,lang_Portuguese
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,INR,83.00,2.0,Negative,...,4,15,1,6,2,0,0,0,1,0
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,BRL,5.70,4.0,Positive,...,4,15,3,2,3,2,0,0,0,1
2,3,Pahal Balay,27,Google,Pixel 6,864.53,INR,83.00,4.0,Positive,...,1,10,0,6,2,2,0,0,1,0
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,AED,3.67,3.0,Positive,...,6,17,1,0,0,2,1,0,0,0
4,5,Yago Leão,38,Motorola,Edge 50,792.13,BRL,5.70,3.0,Neutral,...,2,0,2,2,3,1,0,0,0,1


In [19]:
df_clean.head()

,review_id,customer_name,age,brand,model,price_usd,currency,exchange_rate_to_usd,rating,sentiment,...,language,review_date,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,helpful_votes,source
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,INR,83.00,2.0,Negative,...,Hindi,11/6/2023,True,1,1,3,2,1,1,Amazon
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,BRL,5.70,4.0,Positive,...,Portuguese,3/30/2023,True,3,2,4,3,2,5,Flipkart
2,3,Pahal Balay,27,Google,Pixel 6,864.53,INR,83.00,4.0,Positive,...,Hindi,12/7/2022,True,3,5,3,2,4,8,AliExpress
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,AED,3.67,3.0,Positive,...,English,3/11/2025,False,1,3,2,1,2,3,Amazon
4,5,Yago Leão,38,Motorola,Edge 50,792.13,BRL,5.70,3.0,Neutral,...,Portuguese,9/29/2023,True,3,3,2,2,1,0,BestBuy
